# Reproduce PaiNN Results
This notebook loads the best model checkpoint and its configuration to regenerate metrics on the QM9 test set.

In [ ]:
import os
import json
import torch
import torch.nn as nn
from torch_geometric.datasets import QM9
from torch_geometric.loader import DataLoader
from torch_geometric.utils import scatter
from torch_geometric.nn import radius_graph
from tqdm.auto import tqdm

device = torch.device('cpu')
print(f"Using device: {device}")

## 1. Model Definition

In [ ]:
class GaussianRBF(nn.Module):
    def __init__(self, num_rbf=20, cutoff=5.0):
        super().__init__()
        centers = torch.linspace(0.0, cutoff, num_rbf)
        self.register_buffer('centers', centers)
        self.width = (cutoff / num_rbf) ** 2

    def forward(self, dist):
        return torch.exp(-((dist.unsqueeze(-1) - self.centers) ** 2) / self.width)


class CosineCutoff(nn.Module):
    def __init__(self, cutoff=5.0):
        super().__init__()
        self.cutoff = cutoff

    def forward(self, dist):
        return torch.where(
            dist < self.cutoff,
            0.5 * (torch.cos(dist * torch.pi / self.cutoff) + 1.0),
            torch.zeros_like(dist)
        )


class PaiNNMessage(nn.Module):
    def __init__(self, hidden_dim, num_rbf=20, cutoff=5.0):
        super().__init__()
        self.rbf        = GaussianRBF(num_rbf, cutoff)
        self.cutoff     = CosineCutoff(cutoff)
        self.filter_net = nn.Sequential(
            nn.Linear(num_rbf, hidden_dim), nn.SiLU(),
            nn.Linear(hidden_dim, 3 * hidden_dim),
        )
        self.scalar_net = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim), nn.SiLU(),
            nn.Linear(hidden_dim, 3 * hidden_dim),
        )

    def forward(self, s, v, edge_index, pos):
        row, col   = edge_index
        diff       = pos[row] - pos[col]
        dist       = diff.norm(dim=-1, keepdim=True).clamp(min=1e-8)
        unit       = diff / dist
        dist       = dist.squeeze(-1)
        rbf_out    = self.rbf(dist)
        envelope   = self.cutoff(dist).unsqueeze(-1)
        filter_out = self.filter_net(rbf_out) * envelope
        W1, W2, W3 = filter_out.chunk(3, dim=-1)
        phi        = self.scalar_net(s[col])
        P1, P2, P3 = phi.chunk(3, dim=-1)
        ds_msg     = P1 * W1
        v_src      = v[col]
        v_msg1     = v_src * (P2 * W2).unsqueeze(1)
        v_msg2     = unit.unsqueeze(-1) * (P3 * W3).unsqueeze(1)
        dv_msg     = v_msg1 + v_msg2
        ds         = scatter(ds_msg, row, dim=0, dim_size=s.size(0), reduce='sum')
        dv         = scatter(dv_msg, row, dim=0, dim_size=s.size(0), reduce='sum')
        return s + ds, v + dv


class PaiNNUpdate(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()
        self.U          = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.V          = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.update_net = nn.Sequential(
            nn.Linear(2 * hidden_dim, hidden_dim), nn.SiLU(),
            nn.Linear(hidden_dim, 3 * hidden_dim),
        )

    def forward(self, s, v):
        Uv      = torch.einsum('nch,hd->ncd', v, self.U.weight)
        Vv      = torch.einsum('nch,hd->ncd', v, self.V.weight)
        vv_norm = (Vv * Vv).sum(dim=1)
        a       = self.update_net(torch.cat([s, vv_norm], dim=-1))
        a1, a2, a3 = a.chunk(3, dim=-1)
        ds      = a1 + a3 * (Uv * Vv).sum(dim=1)
        dv      = Uv * a2.unsqueeze(1)
        return s + ds, v + dv


class PaiNNModel(nn.Module):
    def __init__(self, hidden_dim=128, n_layers=3, num_rbf=20,
                 cutoff=5.0, max_neighbors=32):
        super().__init__()
        self.cutoff         = cutoff
        self.max_neighbors  = max_neighbors
        self.embedding      = nn.Embedding(10, hidden_dim)
        self.message_layers = nn.ModuleList([
            PaiNNMessage(hidden_dim, num_rbf, cutoff) for _ in range(n_layers)
        ])
        self.update_layers  = nn.ModuleList([
            PaiNNUpdate(hidden_dim) for _ in range(n_layers)
        ])
        self.readout = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 2), nn.SiLU(),
            nn.Linear(hidden_dim // 2, 1),
        )

    def forward(self, z, pos, batch):
        edge_index = radius_graph(
            pos, r=self.cutoff, batch=batch,
            max_num_neighbors=self.max_neighbors
        )
        s = self.embedding(z)
        v = torch.zeros(s.size(0), 3, s.size(1), device=s.device, dtype=s.dtype)
        for msg, upd in zip(self.message_layers, self.update_layers):
            s, v = msg(s, v, edge_index, pos)
            s, v = upd(s, v)
        atom_out = self.readout(s).squeeze(-1)
        return scatter(atom_out, batch, dim=0, reduce='sum')

## 2. Load Configuration and Initialize Model

In [ ]:
config_path = 'painn_config.json'
model_path  = 'painn_best_model.pt'

if not os.path.exists(config_path) or not os.path.exists(model_path):
    raise FileNotFoundError("Required model or config file missing. Please run training first.")

with open(config_path, 'r') as f:
    config = json.load(f)

model = PaiNNModel(
    hidden_dim=config['hidden_dim'],
    n_layers=config['n_layers'],
    num_rbf=config['num_rbf'],
    cutoff=config['cutoff'],
    max_neighbors=config['max_neighbors'],
).to(device)

model.load_state_dict(torch.load(model_path, map_location=device))
model.eval()
print("Model and configuration loaded successfully.")

## 3. Load Dataset

In [ ]:
dataset      = QM9(root='./data/QM9')
test_dataset = dataset[118000:]
test_loader  = DataLoader(test_dataset, batch_size=16, shuffle=False)

target_mean = config['target_mean']
target_mad  = config['target_mad']
TARGET_IDX  = config['target_idx']

def denormalize(y_norm):
    return y_norm * target_mad + target_mean

print(f"Test set: {len(test_dataset)} molecules")

## 4. Evaluate and Report

In [ ]:
@torch.no_grad()
def evaluate(model, loader):
    total_mae = 0.0
    for batch in tqdm(loader, desc='Evaluating'):
        batch     = batch.to(device)
        preds     = denormalize(model(batch.z, batch.pos, batch.batch))
        total_mae += (preds - batch.y[:, TARGET_IDX]).abs().sum().item()
    return total_mae / len(loader.dataset)

test_mae = evaluate(model, test_loader)
print(f"Reproduced Test MAE: {test_mae * 1000:.2f} meV")